In [31]:
import os
import shutil

# Define the current working directory
current_directory = os.getcwd()

print(f"Deleting all files and directories in: {current_directory} (excluding /content/sample_data)")

# Iterate over all items in the current directory
for item in os.listdir(current_directory):
    item_path = os.path.join(current_directory, item)
    # Exclude the common Colab sample_data directory
    if item == 'sample_data':
        print(f"Skipping {item_path}")
        continue

    try:
        if os.path.isfile(item_path):
            os.remove(item_path)
            print(f"Deleted file: {item_path}")
        elif os.path.isdir(item_path):
            shutil.rmtree(item_path)
            print(f"Deleted directory and its contents: {item_path}")
    except Exception as e:
        print(f"Error deleting {item_path}: {e}")

print("Deletion process complete.")

Deleting all files and directories in: /content (excluding /content/sample_data)
Deleted file: /content/cluster_info.json
Deleted file: /content/cluster_info (2).json
Deleted file: /content/cluster_info (3).json
Deleted file: /content/cluster_info (10).json
Deleted file: /content/cluster_info (7).json
Deleted file: /content/cluster_info (9).json
Deleted file: /content/cluster_info (6).json
Deleted file: /content/cluster_info (8).json
Deleted file: /content/cluster_info (4).json
Deleted file: /content/cluster_info (5).json
Deleted file: /content/cluster_summary_complete.csv
Skipping /content/sample_data
Deletion process complete.


In [32]:
# ============================================================
# ESM ATLAS CLUSTER EXTRACTOR - FINAL VERSION
# Upload your cluster_info.json files all at once.
# ============================================================

import json
import pandas as pd
from google.colab import files

# Store all cluster data here
all_clusters = []
cluster_counter = 0

print("="*60)
print("🧬 ESM Atlas Cluster Metadata Extractor")
print("="*60)
print("\n📌 IMPORTANT INSTRUCTIONS:")
print("   1. Click 'Choose Files' and select ALL your 'cluster_info.json' files simultaneously.")
print("      (e.g., hold Ctrl/Cmd and click each file, or Shift to select a range)")
print("   2. The script will process all selected files automatically.")
print("============================================================")

print("\n📤 --- Please upload your cluster_info.json files now ---")

# Upload all files at once
uploaded = files.upload()

# Check if any files were uploaded
if not uploaded:
    print("❌ No files uploaded. Please re-run the cell and try again.")
else:
    print(f"✅ {len(uploaded)} files selected for processing.")

    for file_name, file_content in uploaded.items():
        cluster_counter += 1
        print(f"\nProcessing file {cluster_counter}/{len(uploaded)}: '{file_name}'")

        try:
            # Parse JSON
            content = file_content.decode('utf-8')
            data = json.loads(content)

            # Extract proteins list
            proteins = data.get("proteins", [])
            num_proteins = len(proteins)

            # Extract cluster information
            clusters = data.get("clusters", {})

            for cluster_id, cluster_info in clusters.items():
                # Get cluster name (fallback to hash if missing)
                cluster_name = cluster_info.get("cluster_name")
                if not cluster_name:
                    cluster_name = cluster_id

                # Get PFAM domains as a readable string
                pfam_domains = cluster_info.get("cluster_top_pfam_domains", {})
                pfam_string = "; ".join([
                    f"{v.get('name', k)} ({v.get('count', 0)})"
                    for k, v in pfam_domains.items()
                ])

                # Build the entry
                cluster_entry = {
                    "cluster_id": cluster_id,
                    "cluster_name": cluster_name,
                    "num_proteins": num_proteins,
                    "mean_domain_coverage": cluster_info.get("cluster_mean_domain_coverage", 0),
                    "pct_characterized": cluster_info.get("cluster_pct_characterized", 0),
                    "taxonomy_rank": cluster_info.get("cluster_taxonomy_info", {}).get("rank", "Unknown"),
                    "taxonomy_name": cluster_info.get("cluster_taxonomy_info", {}).get("name", "Unknown"),
                    "pfam_domains": pfam_string
                }

                all_clusters.append(cluster_entry)

                # Print summary
                print(f"\n✅ EXTRACTED: {cluster_entry['cluster_name'][:50]}...")
                print(f"   📁 Proteins in cluster: {num_proteins}")
                print(f"   📊 Domain Coverage: {cluster_entry['mean_domain_coverage']:.2f}")

        except json.JSONDecodeError:
            print(f"❌ Error: '{file_name}' is not a valid JSON file. Skipping...")
        except Exception as e:
            print(f"❌ Error processing '{file_name}': {e}")

# ============================================================
# FINAL STEP: EXPORT TO CSV AND DOWNLOAD
# ============================================================

if all_clusters:
    print("\n" + "="*60)
    print(f"✅ Successfully extracted data from {len(all_clusters)} clusters!")
    print("============================================================")

    # Create DataFrame
    df = pd.DataFrame(all_clusters)

    # Reorder columns for readability
    column_order = [
        "cluster_name",
        "num_proteins",
        "pfam_domains",
        "cluster_id",
        "mean_domain_coverage",
        "pct_characterized",
        "taxonomy_name",
        "taxonomy_rank"
    ]
    df = df[column_order]

    # Display the table
    print("\n📊 SUMMARY TABLE:")
    print(df.to_string(index=False))

    # Save to CSV
    output_filename = "cluster_summary_complete.csv"
    df.to_csv(output_filename, index=False)

    # Download the CSV
    print(f"\n⬇️ Downloading '{output_filename}'...")
    files.download(output_filename)

    print("\n🎉 COMPLETE! The CSV file has been downloaded to your computer.")
    print("   Open it in Excel or Google Sheets to see your data.")
else:
    print("\n❌ No data was extracted. Please ensure you uploaded your 'cluster_info.json' files.")

🧬 ESM Atlas Cluster Metadata Extractor

📌 IMPORTANT INSTRUCTIONS:
   1. Click 'Choose Files' and select ALL your 'cluster_info.json' files simultaneously.
      (e.g., hold Ctrl/Cmd and click each file, or Shift to select a range)
   2. The script will process all selected files automatically.

📤 --- Please upload your cluster_info.json files now ---


Saving cluster_info (2).json to cluster_info (2).json
Saving cluster_info (3).json to cluster_info (3).json
Saving cluster_info (4).json to cluster_info (4).json
Saving cluster_info (5).json to cluster_info (5).json
Saving cluster_info (6).json to cluster_info (6).json
Saving cluster_info (7).json to cluster_info (7).json
Saving cluster_info (8).json to cluster_info (8).json
Saving cluster_info (9).json to cluster_info (9).json
Saving cluster_info (10).json to cluster_info (10).json
Saving cluster_info.json to cluster_info.json
✅ 10 files selected for processing.

Processing file 1/10: 'cluster_info (2).json'

✅ EXTRACTED: 08fce4fbede299ab48f88e752fbecd4a...
   📁 Proteins in cluster: 50
   📊 Domain Coverage: 0.45

Processing file 2/10: 'cluster_info (3).json'

✅ EXTRACTED: cbd2d143249c8a7ae0f56ac1a24cc12f...
   📁 Proteins in cluster: 95
   📊 Domain Coverage: 0.58

Processing file 3/10: 'cluster_info (4).json'

✅ EXTRACTED: 1475a16d26c7ee98cff0ee07d712039e...
   📁 Proteins in cluster: 6

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 COMPLETE! The CSV file has been downloaded to your computer.
   Open it in Excel or Google Sheets to see your data.
